# NB_patient_360

Builds `hcp_databricks_workspace.gold.patient_360` per the STTM (Silver to Gold, v1.0): one row per patient, driven by `silver.patients` with LEFT JOINs to hospitals, insurance providers and per-patient lab result aggregates. Load type: FULL rebuild (CREATE OR REPLACE TABLE ... AS SELECT).

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS hcp_databricks_workspace.gold

In [0]:
%sql
CREATE OR REPLACE TABLE hcp_databricks_workspace.gold.patient_360
COMMENT 'Gold: one row per patient with hospital, insurance and lab result summary (STTM silver to gold v1.0)'
AS
WITH lab_agg AS (
  -- Per-patient lab result aggregates
  SELECT
    patient_id,
    COUNT(*)                                   AS total_tests,
    COUNT_IF(result_status = 'Abnormal')       AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')       AS critical_test_count,
    MAX(test_date)                             AS last_test_date
  FROM hcp_databricks_workspace.silver.lab_results
  GROUP BY patient_id
)
SELECT
  p.patient_id                                                              AS patient_id,
  p.first_name                                                              AS first_name,
  p.last_name                                                               AS last_name,
  CAST(p.date_of_birth AS DATE)                                             AS date_of_birth,   -- silver column is TIMESTAMP
  CAST(floor(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender                                                                  AS gender,
  p.city                                                                    AS city,
  p.state                                                                   AS state,
  p.registered_hospital_id                                                  AS hospital_id,
  h.hospital_name                                                           AS hospital_name,
  p.insurance_provider_id                                                   AS insurance_provider_id,
  ip.provider_name                                                          AS insurance_provider_name,
  CAST(coalesce(la.total_tests, 0) AS INT)                                  AS total_tests,
  CAST(coalesce(la.abnormal_test_count, 0) AS INT)                          AS abnormal_test_count,
  CAST(coalesce(la.critical_test_count, 0) AS INT)                          AS critical_test_count,
  la.last_test_date                                                         AS last_test_date
FROM hcp_databricks_workspace.silver.patients p
LEFT JOIN hcp_databricks_workspace.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN hcp_databricks_workspace.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg la
  ON p.patient_id = la.patient_id

In [0]:
%sql
-- Grain and reconciliation checks against silver
SELECT
  (SELECT count(*) FROM hcp_databricks_workspace.silver.patients)            AS silver_patients,
  count(*)                                                                   AS gold_rows,
  count(DISTINCT patient_id)                                                 AS distinct_patients,
  sum(total_tests)                                                           AS gold_total_tests,
  (SELECT count(*) FROM hcp_databricks_workspace.silver.lab_results
    WHERE patient_id IN (SELECT patient_id FROM hcp_databricks_workspace.silver.patients)) AS silver_matched_tests,
  sum(abnormal_test_count)                                                   AS abnormal_tests,
  sum(critical_test_count)                                                   AS critical_tests,
  count_if(hospital_name IS NULL)                                            AS missing_hospital_name,
  count_if(insurance_provider_name IS NULL)                                  AS missing_provider_name,
  min(age)                                                                   AS min_age,
  max(age)                                                                   AS max_age
FROM hcp_databricks_workspace.gold.patient_360

In [0]:
%sql
SELECT * FROM hcp_databricks_workspace.gold.patient_360 ORDER BY patient_id LIMIT 20